# 01 - Data Exploration

This notebook explores the raw market data and geopolitical event database.

**Objectives:**
1. Fetch and visualize stock price data for US, India, China
2. Explore the curated geopolitical event database
3. Basic statistical analysis of each market
4. Cross-market correlation analysis

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from src.data_collection.stock_fetcher import StockDataFetcher
from src.data_collection.tariff_tracker import TariffEventTracker
from src.data_collection.conflict_tracker import ConflictEventTracker
from src.utils.helpers import compute_returns

pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 100)

## 1. Fetch Market Data

In [ ]:
fetcher = StockDataFetcher()

# Fetch main indices
sp500 = fetcher.fetch_symbol('^GSPC', '2015-01-01', '2025-12-31')
nifty = fetcher.fetch_symbol('^NSEI', '2015-01-01', '2025-12-31')
hsi = fetcher.fetch_symbol('^HSI', '2015-01-01', '2025-12-31')

print(f'S&P 500: {len(sp500)} rows, {sp500.index[0]} to {sp500.index[-1]}')
print(f'NIFTY 50: {len(nifty)} rows, {nifty.index[0]} to {nifty.index[-1]}')
print(f'Hang Seng: {len(hsi)} rows, {hsi.index[0]} to {hsi.index[-1]}')

## 2. Normalized Price Comparison

In [ ]:
fig = go.Figure()

for name, data, color in [
    ('S&P 500', sp500, '#1976d2'),
    ('NIFTY 50', nifty, '#e65100'),
    ('Hang Seng', hsi, '#c62828'),
]:
    if not data.empty:
        normalized = (data['close'] / data['close'].iloc[0]) * 100
        fig.add_trace(go.Scatter(x=normalized.index, y=normalized.values, name=name, line=dict(color=color, width=2)))

fig.update_layout(title='US-India-China Market Comparison (Base=100)', yaxis_title='Normalized Price', template='plotly_white', height=500)
fig.show()

## 3. Geopolitical Event Database

In [ ]:
# Load curated events
tariff_tracker = TariffEventTracker()
tariff_events = tariff_tracker.get_curated_tariff_events()

conflict_tracker = ConflictEventTracker()
conflict_events = conflict_tracker.get_curated_conflict_events()

all_events = conflict_tracker.get_combined_geopolitical_events()

print(f'Tariff events: {len(tariff_events)}')
print(f'Conflict events: {len(conflict_events)}')
print(f'Total events: {len(all_events)}')

all_events.head(10)

## 4. Return Statistics

In [ ]:
stats = {}
for name, data in [('S&P 500', sp500), ('NIFTY 50', nifty), ('Hang Seng', hsi)]:
    if not data.empty:
        returns = compute_returns(data['close'])
        stats[name] = {
            'Ann. Return': returns.mean() * 252,
            'Ann. Volatility': returns.std() * np.sqrt(252),
            'Sharpe Ratio': (returns.mean() * 252) / (returns.std() * np.sqrt(252)),
            'Max Drawdown': ((data['close'] / data['close'].cummax()) - 1).min(),
            'Skewness': returns.skew(),
            'Kurtosis': returns.kurtosis(),
        }

pd.DataFrame(stats).round(4)

## 5. Cross-Market Correlation

In [ ]:
returns_dict = {}
for name, data in [('US', sp500), ('India', nifty), ('China', hsi)]:
    if not data.empty:
        ret = compute_returns(data['close'])
        ret.index = ret.index.normalize()
        returns_dict[name] = ret

returns_df = pd.DataFrame(returns_dict).dropna()
print(f'Aligned return data: {len(returns_df)} rows')
print('\nCorrelation Matrix:')
returns_df.corr().round(4)